# Sample Eddy Tilt

Focused plots for a small set of representative vertically checked eddies.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths)

sample_eddies = [75, 463, 943, 1988, 31, 221, 1766, 2391]
colors = plt.cm.tab10(np.linspace(0, 1, len(sample_eddies)))
df_eddies.loc[df_eddies.Eddy.isin(sample_eddies), ["Eddy", "Cyc", "Day", "TiltDis", "TiltDir"]].head()


In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
for ax, eddy, color in zip(axs.ravel(), sample_eddies, colors):
    df = df_eddies.loc[df_eddies.Eddy == eddy].sort_values("Day")
    if df.empty:
        ax.set_axis_off()
        continue
    ax.plot(df.Day, df.TiltDis, color=color, lw=1.8)
    ax.scatter(df.Day, df.TiltDis, color=color, s=12)
    ax.set_title(f"{df.Cyc.iloc[0]}{eddy}")
    ax.set_xlabel("Day")
    ax.set_ylabel("Tilt distance (km)")
    ax.grid(True, axis="y", alpha=0.25)
plt.show()


In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
for ax, eddy, color in zip(axs.ravel(), sample_eddies, colors):
    df = df_eddies.loc[df_eddies.Eddy == eddy].sort_values("Day")
    if df.empty:
        ax.set_axis_off()
        continue
    ax.contourf(grid.X_grid, grid.Y_grid, np.where(grid.mask_rho, grid.h / 1e3, np.nan), cmap="gray", alpha=0.65)
    ax.plot(df.xc, df.yc, color=color, lw=1.5)
    for row in df.itertuples():
        if np.isfinite(row.TiltDis) and np.isfinite(row.TiltDir):
            xb, yb = tilt.point_from_bearing((row.xc, row.yc), row.TiltDis, row.TiltDir)
            ax.plot([row.xc, xb], [row.yc, yb], color=color, lw=0.8, alpha=0.55)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(df.xc.min() - 10, df.xc.max() + 10)
    ax.set_ylim(df.yc.min() - 10, df.yc.max() + 10)
    ax.set_title(f"{df.Cyc.iloc[0]}{eddy}")
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
plt.show()


In [ ]:
mag_bins = [0, 10, 20, 30, 40, np.inf]
fig, axs = plt.subplots(2, 4, figsize=(13, 7), subplot_kw={"projection": "polar"}, constrained_layout=True)
for ax, eddy in zip(axs.ravel(), sample_eddies):
    df = df_eddies.loc[df_eddies.Eddy == eddy].dropna(subset=["TiltDir", "TiltDis"])
    if df.empty:
        ax.set_axis_off()
        continue
    colors = plt.cm.Reds(np.linspace(0.15, 1, len(mag_bins) - 1)) if df.Cyc.iloc[0] == "AE" else plt.cm.Blues(np.linspace(0.15, 1, len(mag_bins) - 1))
    tilt.plot_windrose(ax, df, title=f"{df.Cyc.iloc[0]}{eddy}", mag_bins=mag_bins, colors=colors)
axs.ravel()[0].legend(title="Tilt dist. (km)", loc="upper left", bbox_to_anchor=(1.05, 1.05), frameon=False)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
data = []
labels = []
for eddy in sample_eddies:
    df = df_eddies.loc[df_eddies.Eddy == eddy]
    vals = df.TiltDis.dropna().to_numpy()
    if vals.size:
        data.append(vals)
        labels.append(f"{df.Cyc.iloc[0]}{eddy}")
ax.boxplot(data, tick_labels=labels, showfliers=True, patch_artist=True)
ax.set_ylabel("Tilt distance (km)")
ax.grid(True, axis="y", alpha=0.25)
plt.show()
